In [1]:
### Strategy explanation

# Why we simulate territories
The Kaggle dataset has real drug sales over time but no territory or
rep activity data (those are proprietary in real pharma companies).

Our approach:
  Real Rx volume  ←  from salesmonthly.csv
  Territory ID    ←  we assign 200 territories, each with monthly sales
  Rep features    ←  simulated with realistic distributions calibrated
                     to the actual sales values

This is exactly how real pharma DS teams work when field CRM data
is under NDA. We state this transparently in the README.

In [5]:
# Load monthly data and pick target drug

In [7]:
import pandas as pd
import numpy as np

DRUG_COLS = ["M01AB", "M01AE", "N02BA", "N02BE", "N05B", "N05C", "R03", "R06"]
DRUG_NAMES = {
    "M01AB":"Anti-inflammatory","M01AE":"Ibuprofen-class",
    "N02BA":"Aspirin-class","N02BE":"Paracetamol",
    "N05B":"Anxiolytics","N05C":"Hypnotics",
    "R03":"Respiratory","R06":"Antihistamines"
}

monthly = pd.read_csv("C:/Users/Hello User/Downloads/salesmonthly.csv", parse_dates=["datum"])
monthly["year"]  = monthly["datum"].dt.year
monthly["month_num"] = monthly["datum"].dt.month

# We'll model N02BE (Paracetamol) — highest volume drug, best for SFE demo
# You can change this to any drug code
TARGET_DRUG = "N02BE"
print(f"Target drug: {DRUG_NAMES[TARGET_DRUG]} ({TARGET_DRUG})")
print(f"Monthly records available: {len(monthly)}")
print(f"\nSales summary:")
print(monthly[TARGET_DRUG].describe().round(2))

Target drug: Paracetamol (N02BE)
Monthly records available: 70

Sales summary:
count      70.00
mean      892.54
std       338.84
min         0.00
25%       648.19
50%       865.82
75%      1061.58
max      1856.81
Name: N02BE, dtype: float64


In [11]:
import os

# Create folders if they don't exist
os.makedirs("data", exist_ok=True)       # creates data/ folder
os.makedirs("outputs", exist_ok=True)    # creates outputs/ folder too (needed later)

print("Folders ready ✓")

Folders ready ✓


In [1]:
#cell 3

In [5]:
import os

# Search for salesmonthly.csv on your system
print("Searching for salesmonthly.csv...\n")

# Check common locations
search_paths = [
    ".",                          # current folder
    "..",                         # one folder up
    "data/",                      # data subfolder
    os.path.expanduser("~"),      # home directory
    "C:/Users",                   # Windows users folder
]

found_paths = []
for root, dirs, files in os.walk("C:/"):
    for file in files:
        if file in ["salesmonthly.csv", 
                    "salesweekly.csv", 
                    "salesdaily.csv"]:
            full_path = os.path.join(root, file)
            found_paths.append(full_path)
            print(f"✅ Found: {full_path}")

if not found_paths:
    print("❌ Files not found — check Downloads folder manually")

# Also print current working directory
print(f"\nYour notebook is running from:")
print(os.getcwd())

Searching for salesmonthly.csv...

✅ Found: C:/Users\Hello User\Downloads\salesdaily.csv
✅ Found: C:/Users\Hello User\Downloads\salesmonthly.csv
✅ Found: C:/Users\Hello User\Downloads\salesweekly.csv

Your notebook is running from:
C:\Users\Hello User


In [13]:
import shutil
import os

SOURCE_FOLDER = r"C:\Users\Hello User\Downloads"
DEST_FOLDER   = os.getcwd()

for filename in ["salesdaily.csv",
                 "salesweekly.csv",
                 "salesmonthly.csv"]:
    src  = os.path.join(SOURCE_FOLDER, filename)
    dest = os.path.join(DEST_FOLDER,   filename)
    shutil.copy(src, dest)
    print(f"✅ Copied: {filename} → {DEST_FOLDER}")

print(f"\n📁 All 3 files now in: {DEST_FOLDER}")
print("✅")

✅ Copied: salesdaily.csv → C:\Users\Hello User
✅ Copied: salesweekly.csv → C:\Users\Hello User
✅ Copied: salesmonthly.csv → C:\Users\Hello User

📁 All 3 files now in: C:\Users\Hello User
✅


In [9]:
import pandas as pd
import numpy as np
import os

os.makedirs("data",    exist_ok=True)
os.makedirs("outputs", exist_ok=True)

monthly = pd.read_csv("salesmonthly.csv", parse_dates=["datum"])
TARGET_DRUG = "N02BE"

np.random.seed(42)

N_TERRITORIES = 200
territory_ids = [f"TER_{str(i).zfill(3)}" 
                 for i in range(1, N_TERRITORIES + 1)]

rows = []
for t_id in territory_ids:

    # ── Territory fixed traits ────────────────────────────
    # hcp_tier drives base potential — 1=best doctors
    hcp_tier         = np.random.choice([1, 2, 3], p=[0.2, 0.5, 0.3])
    rep_tenure_years = np.random.randint(1, 10)
    market_access    = np.random.randint(3, 10)

    # ── Base Rx set by hcp_tier + market_access ───────────
    # NOT by raw sales file — this is the key fix
    # Tier 1 territories have higher base potential
    tier_base = {1: 800, 2: 500, 3: 250}
    base_rx   = (tier_base[hcp_tier]
                 + market_access * 30
                 + np.random.normal(0, 50))
    base_rx   = max(base_rx, 100)

    for _, month_row in monthly.iterrows():

        # ── Rep activity — varies monthly ─────────────────
        # More experienced reps make more calls
        calls = max(1, int(np.random.poisson(
            lam=6 + rep_tenure_years * 0.4)))

        samples_dropped = max(0, int(
            calls * 1.5 + np.random.normal(0, 2)))

        digital_touches = max(0, int(
            np.random.poisson(lam=calls * 0.6)))

        # Competitor share — negatively impacts Rx
        competitor_share = float(np.clip(
            np.random.beta(2, 3), 0.1, 0.85))

        # Seasonality from real data
        seasonality_idx = float(
            1 + 0.12 * np.sin(
                2 * np.pi * month_row["datum"].month / 12))

        # ── Rx volume: rep features are PRIMARY drivers ────
        # This is the key fix — commercial features drive Rx
        rx_volume = (
            # Base potential from territory quality
            base_rx * 0.3

            # Rep call effect — strongest driver
            + calls * (20 - hcp_tier * 3)

            # Samples effect
            + samples_dropped * 8

            # Digital touchpoints
            + digital_touches * 5

            # Competitor hurts sales
            - competitor_share * 300

            # Market access boosts sales
            + market_access * 20

            # Rep tenure boosts efficiency
            + rep_tenure_years * 15

            # Seasonality
            + base_rx * 0.1 * (seasonality_idx - 1)

            # Noise — kept small so signal is clear
            + np.random.normal(0, 30)
        )
        rx_volume = float(max(rx_volume, 10))

        rows.append({
            "territory_id"    : t_id,
            "datum"           : month_row["datum"],
            "year"            : int(month_row["datum"].year),
            "month"           : int(month_row["datum"].month),
            "hcp_tier"        : int(hcp_tier),
            "rep_tenure"      : int(rep_tenure_years),
            "calls"           : int(calls),
            "samples_dropped" : int(samples_dropped),
            "digital_touches" : int(digital_touches),
            "competitor_share": round(float(competitor_share), 3),
            "market_access"   : int(market_access),
            "seasonality_idx" : round(float(seasonality_idx), 3),
            "base_rx"         : round(float(base_rx), 2),
            "rx_volume"       : round(float(rx_volume), 2)
        })

master_df = pd.DataFrame(rows)
master_df.to_csv("data/sfe_master_dataset.csv", index=False)

print(f"✅ master_df saved!")
print(f"   Shape      : {master_df.shape}")
print(f"   Territories: {master_df['territory_id'].nunique()}")
print(f"   Months     : {master_df['datum'].nunique()}")
print(f"\nRx volume stats:")
print(master_df["rx_volume"].describe().round(2))

# Quick correlation check — calls should correlate with rx
corr = master_df[["calls","samples_dropped","digital_touches",
                  "competitor_share","market_access",
                  "hcp_tier","rep_tenure","rx_volume"]].corr()
print(f"\nCorrelation with rx_volume:")
print(corr["rx_volume"].drop("rx_volume").sort_values(
      ascending=False).round(3))

✅ master_df saved!
   Shape      : (14000, 14)
   Territories: 200
   Months     : 70

Rx volume stats:
count    14000.00
mean       518.48
std        158.41
min         10.00
25%        410.64
50%        520.96
75%        625.71
max       1173.42
Name: rx_volume, dtype: float64

Correlation with rx_volume:
calls               0.631
samples_dropped     0.625
rep_tenure          0.450
digital_touches     0.445
market_access       0.382
competitor_share   -0.359
hcp_tier           -0.464
Name: rx_volume, dtype: float64


In [15]:
#cell 4-Territory-level aggregate (for clustering)

In [11]:
territory_agg = master_df.groupby("territory_id").agg(
    avg_calls         = ("calls",            "mean"),
    avg_samples       = ("samples_dropped",  "mean"),
    avg_digital       = ("digital_touches",  "mean"),
    avg_competitor    = ("competitor_share", "mean"),
    avg_market_access = ("market_access",    "mean"),
    avg_rx            = ("rx_volume",        "mean"),
    total_rx          = ("rx_volume",        "sum"),
    hcp_tier          = ("hcp_tier",         "first"),
    rep_tenure        = ("rep_tenure",       "first")
).reset_index()

territory_agg["rx_per_call"] = (territory_agg["avg_rx"] /
                                 territory_agg["avg_calls"])

territory_agg.to_csv("data/sfe_territory_agg.csv", index=False)
print(f"✅ territory_agg saved: {territory_agg.shape}")

✅ territory_agg saved: (200, 11)
